In [1]:
import sys
import pandas as pd
import numpy as np
from tqdm import tqdm

sys.path.append("..")
from data.base import get_connection

print("Dependencies loaded successfully.")

Dependencies loaded successfully.


In [2]:
# Aggregate tickets by week, camera type, and intersection
query = """
    SELECT 
        CAST(violation_code AS INTEGER) AS violation_code,
        CASE 
            WHEN CAST(violation_code AS INTEGER) = 7 THEN 'Red Light'
            WHEN CAST(violation_code AS INTEGER) = 36 THEN 'Speed'
        END AS camera_type,
        street_name,
        intersecting_street,
        DATE_TRUNC('week', CAST(issue_date AS TIMESTAMP)) AS ticket_week,
        COUNT(*) AS weekly_tickets
    FROM tickets_all
    WHERE CAST(violation_code AS INTEGER) IN (7, 36)
      AND street_name IS NOT NULL 
      AND intersecting_street IS NOT NULL
      AND issue_date >= '2016-01-01'
    GROUP BY 1, 2, 3, 4, 5
    ORDER BY street_name, intersecting_street, camera_type, ticket_week
"""

con = get_connection()
try:
    weekly_raw = con.sql(query).df()
finally:
    con.close()

# Apply standard cleaning to build exact matching search_query
raw_text = weekly_raw['street_name'].fillna('') + ' ' + weekly_raw['intersecting_street'].fillna('')
weekly_raw['search_query'] = (
    raw_text
    .str.upper()
    .str.replace(r'\b(NB|SB|EB|WB|N/B|S/B|E/B|W/B)\b', '', regex=True)
    .str.replace(r'\(.*?\)', '', regex=True)
    .str.replace('@', ' AND ')
    .str.replace(r'^\s+', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    + ', NEW YORK, NY'
)

# Keep only intersections that successfully geocoded in yearly_camera_coordinates.csv
valid_coords = pd.read_csv('../data/yearly_camera_coordinates.csv')[
    ['search_query', 'Latitude', 'Longitude']
].drop_duplicates(subset=['search_query'])

weekly_tickets = weekly_raw.merge(valid_coords, on='search_query', how='inner')
weekly_tickets['ticket_week'] = pd.to_datetime(weekly_tickets['ticket_week'])

print(f"Aggregated records: {len(weekly_tickets):,} weekly observations across valid geocoded cameras.")
weekly_tickets.head()

Aggregated records: 739,583 weekly observations across valid geocoded cameras.


,violation_code,camera_type,street_name,intersecting_street,ticket_week,weekly_tickets,search_query,Latitude,Longitude
0,7,Red Light,10TH AVE (N/B) @ WES,T 34TH ST,2016-09-19,9,"10TH AVE AND WES T 34TH ST, NEW YORK, NY",40.75459,-73.999149
1,7,Red Light,10TH AVE (N/B) @ WES,T 34TH ST,2016-09-26,8,"10TH AVE AND WES T 34TH ST, NEW YORK, NY",40.75459,-73.999149
2,7,Red Light,10TH AVE (N/B) @ WES,T 34TH ST,2016-10-03,14,"10TH AVE AND WES T 34TH ST, NEW YORK, NY",40.75459,-73.999149
3,7,Red Light,10TH AVE (N/B) @ WES,T 34TH ST,2016-10-10,9,"10TH AVE AND WES T 34TH ST, NEW YORK, NY",40.75459,-73.999149
4,7,Red Light,10TH AVE (N/B) @ WES,T 34TH ST,2016-10-17,10,"10TH AVE AND WES T 34TH ST, NEW YORK, NY",40.75459,-73.999149


In [3]:
def infer_activation_date(group, min_consecutive_active_weeks=3, min_weekly_tickets=5):
    """
    Identifies the go-live changepoint for an enforcement camera.
    
    A camera is considered activated on the earliest date it exhibits sustained
    enforcement volume (at least `min_weekly_tickets` across `min_consecutive_active_weeks`),
    eliminating early test bursts or sporadic miscoded tickets.
    """
    group = group.sort_values('ticket_week').reset_index(drop=True)
    
    # Generate full weekly timeline from 2016-01-01 to latest observed week
    full_weeks = pd.date_range(
        start='2016-01-01', 
        end=group['ticket_week'].max(), 
        freq='W-MON'
    )
    
    ts = pd.DataFrame({'ticket_week': full_weeks})
    ts = ts.merge(group[['ticket_week', 'weekly_tickets']], on='ticket_week', how='left').fillna(0)
    
    # Identify sustained activity window
    is_active = ts['weekly_tickets'] >= min_weekly_tickets
    sustained = is_active.rolling(window=min_consecutive_active_weeks).sum() >= min_consecutive_active_weeks
    
    if sustained.any():
        first_sustained_idx = sustained.idxmax()
        activation_idx = max(0, first_sustained_idx - min_consecutive_active_weeks + 1)
        inferred_date = ts.loc[activation_idx, 'ticket_week']
        confidence = 'high_sustained'
    else:
        # Fallback to the first observed ticket week if volumes never meet the strict rolling threshold
        inferred_date = group['ticket_week'].min()
        confidence = 'low_fallback'
        
    return pd.Series({
        'inferred_activation_date': inferred_date,
        'first_ticket_date': group['ticket_week'].min(),
        'last_ticket_date': group['ticket_week'].max(),
        'total_tickets_issued': group['weekly_tickets'].sum(),
        'detection_confidence': confidence
    })

In [4]:
tqdm.pandas(desc="Inferring Activation Dates")

camera_activations = (
    weekly_tickets
    .groupby(['search_query', 'camera_type', 'violation_code', 'Latitude', 'Longitude'], as_index=False)
    .apply(infer_activation_date, include_groups=False)
    .reset_index()
)

print(f"Activation inference completed for {len(camera_activations):,} distinct camera enforcement points.")
camera_activations['activation_year'] = pd.to_datetime(camera_activations['inferred_activation_date']).dt.year

print("\nCamera Activations by Inferred Year and Type:")
print(pd.crosstab(camera_activations['activation_year'], camera_activations['camera_type'], margins=True))

Activation inference completed for 4,610 distinct camera enforcement points.

Camera Activations by Inferred Year and Type:
camera_type      Red Light  Speed   All
activation_year                        
2016                   175    588   763
2017                    11    169   180
2018                     4    219   223
2019                     2    393   395
2020                     1    673   674
2021                     2    618   620
2022                     5    416   421
2023                     1    250   251
2024                     2    355   357
2025                    44    195   239
2026                   370    117   487
All                    617   3993  4610


In [5]:
# Save to CSV for analysis and inspection
output_csv = "../data/camera_activation_dates.csv"
camera_activations.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

# Cache directly as a DuckDB table for spatial joins in 06_spatial_crash_conflation.ipynb
con = get_connection()
try:
    con.execute("CREATE OR REPLACE TABLE camera_activation_dates AS SELECT * FROM camera_activations")
    stored_count = con.execute("SELECT COUNT(*) FROM camera_activation_dates").fetchone()[0]
    print(f"DuckDB table 'camera_activation_dates' created with {stored_count:,} rows.")
finally:
    con.close()

camera_activations.sample(5)

Saved: ../data/camera_activation_dates.csv
DuckDB table 'camera_activation_dates' created with 4,610 rows.


,index,search_query,camera_type,violation_code,Latitude,Longitude,inferred_activation_date,first_ticket_date,last_ticket_date,total_tickets_issued,detection_confidence,activation_year
2054,2054,"FARMERS BLVD AND 12 0TH AVE, NEW YORK, NY",Red Light,7,40.686457,-73.761171,2026-01-26,2026-01-26,2026-07-20,2240,high_sustained,2026
671,671,"75TH AVE AND BELL B LVD, NEW YORK, NY",Speed,36,40.738717,-73.757342,2020-09-28,2020-09-28,2024-10-28,1642,high_sustained,2020
3777,3777,"S CONDUIT AVE AND 7 9TH ST, NEW YORK, NY",Speed,36,40.673616,-73.857186,2023-09-04,2023-09-04,2026-08-24,56675,high_sustained,2023
552,552,"56 RD. 64 ST - 61 ST, NEW YORK, NY",Speed,36,40.766364,-73.967759,2021-09-20,2021-09-20,2022-01-10,11,low_fallback,2021
9,9,"10 AVE. W22 ST- W 29 ST, NEW YORK, NY",Speed,36,40.591146,-73.983812,2017-01-30,2017-01-30,2020-03-16,73,low_fallback,2017
